# 🏀 Proyecto Machine Learning - Draft NBA 2026 🏀

## ¿Qué es eso del Draft de la NBA?

El **Draft de la NBA** es el proceso anual mediante el cual los 30 equipos de la NBA seleccionan jugadores de universidad, ligas europeas y otras competiciones. Consta de 2 rondas y 60 picks.

La posición en la que eres drafteado determina tu contrato, tu equipo y en muchos casos tu carrera entera.

---
## Punto de partida

Cada año, los equipos NBA seleccionan jugadores en el Draft. Las predicciones tradicionales se basan en opiniones de analistas o de profesionales dedicados a seguir la trayectoria de jóvenes talentos. Sin embargo, me he querido preguntar:

¿Podemos **predecir con Machine Learning** en qué **posición será drafteado un jugador** basándonos en sus datos físicos y estadísticas? En concreto, ¿en qué ronda y posición saldrán seleccionados los españoles Aday Mara, Baba Miller y Sergio De Larrea?

---
## Fundamentos del proyecto

### 1. Definir las predicciones

- ¿En qué **ronda** será drafteado un jugador? 1ª ronda / 2ª ronda / no drafteado (modelo de clasificación)
- ¿En qué rango de elección dentro de su ronda caerá? Por ejemplo: entre el 1 y el 10 o entre el 40 y el 50. (modelo de clasificación)

### 2. ¿Por qué es interesante este proyecto?

Las predicciones actuales se basan en **opiniones subjetivas** de scouts y analistas. Este proyecto propone un enfoque basado en **datos objetivos**: medidas físicas y estadísticas reales de cada jugador.

### 3. ¿A quién le interesa?

Además de al propio aficionado al baloncesto, también puede llamar la atención de un **medio de comunicación deportivo** que quiere publicar un artículo sobre las posibilidades de los jugadores españoles en el Draft 2026 respaldado no por opiniones, sino por un modelo de Machine Learning. Algo distinto y muy actual con el auge de la IA.

---

## Fuentes de datos

### Dataset 1 — NBA Draft histórico
- Todos los jugadores drafteados entre 1989 y 2021.
- Variables: pick, ronda, equipo, estadísticas de carrera NBA.
- Descarga directa desde Kaggle.

### Dataset 2 — NBA Draft Combine
- Jugadores que participaron en el Combine entre los años 2000 y 2026.
- Variables: medidas físicas (altura, envergadura, peso, salto vertical, agilidad, sprint...).
- Obtenido mediante la librería oficial de la NBA para Python.

### ¿Por qué dos datasets?
El Draft histórico nos da la **variable objetivo** (qué pick tuvo cada jugador).  
El Combine nos da las **variables** (qué medidas físicas y de rendimiento tenía antes de ser drafteado).  
Cruzamos ambos por nombre y año para construir el dataset definitivo.

---

In [1]:
import pandas as pd

draft = pd.read_csv('nbaplayersdraft.csv')
combine = pd.read_csv('combine_historico.csv')

print(f"Draft histórico: {draft.shape[0]} jugadores | {draft.shape[1]} variables")
print(f"Combine histórico: {combine.shape[0]} jugadores | {combine.shape[1]} variables")

Draft histórico: 1922 jugadores | 24 variables
Combine histórico: 1873 jugadores | 47 variables


---
## Escaneo inicial de los datos

Antes de modelar, analizamos la estructura real de ambos datasets: columnas disponibles, nulos, solapamiento temporal y viabilidad del merge.

In [ ]:
# Dimensiones y solapamiento temporal
draft_years = set(draft['year'].unique())
combine_years = set(combine['SEASON'].unique())
common_years = sorted(draft_years & combine_years)

print(f"Draft histórico:   {draft.shape[0]} jugadores | {draft.shape[1]} variables | años: {int(draft['year'].min())}–{int(draft['year'].max())}")
print(f"Combine histórico: {combine.shape[0]} jugadores | {combine.shape[1]} variables | años: {int(combine['SEASON'].min())}–{int(combine['SEASON'].max())}")
print(f"Años en común:     {len(common_years)} ({common_years[0]}–{common_years[-1]})") 

### Variables del Draft histórico

| Grupo | Columnas | Uso en el modelo |
|-------|----------|------------------|
| Identificación | `year`, `overall_pick`, `rank`, `team`, `player`, `college` | `overall_pick` es el **target** |
| Stats de carrera NBA | `points_per_game`, `win_shares`, `BPM`, `VORP`... | ⚠️ No usables como features (son posteriores al draft) |

> **Importante:** las estadísticas de carrera reflejan lo que el jugador *hizo después* de ser drafteado. No podemos usarlas para predecir, porque no existirían en el momento de la predicción.

In [ ]:
# Nulos en el dataset de draft
nulos_draft = draft.isnull().sum()
nulos_draft = nulos_draft[nulos_draft > 0].sort_values(ascending=False)
print("Columnas con nulos en draft:")
print(nulos_draft.to_string())
print(f"\nJugadores sin carrera NBA registrada: {draft['games'].isnull().sum()} ({draft['games'].isnull().sum()/len(draft)*100:.1f}%)")
print("→ Drafteados que no llegaron a jugar en la NBA o de los que no hay datos.")

### Variables del Combine

| Grupo | Columnas | Estado |
|-------|----------|--------|
| Físico core | `HEIGHT_WO_SHOES`, `WEIGHT`, `WINGSPAN`, `STANDING_REACH` | ✅ 3% nulos — muy limpias |
| Atletismo | `STANDING_VERTICAL_LEAP`, `MAX_VERTICAL_LEAP`, `LANE_AGILITY_TIME`, `THREE_QUARTER_SPRINT` | ✅ ~14% nulos — usables |
| Morfológico | `BODY_FAT_PCT`, `HAND_LENGTH`, `HAND_WIDTH`, `BENCH_PRESS` | ⚠️ 32–43% nulos — imputar o descartar |
| Tiro en spots | `SPOT_*`, `OFF_DRIB_*`, `ON_MOVE_*` (23 columnas) | ❌ +90% nulos, formato string `"4-5"` — descartar |

In [ ]:
# Análisis de nulos en las features físicas del Combine
cols_fisicas = [
    'HEIGHT_WO_SHOES', 'WEIGHT', 'WINGSPAN', 'STANDING_REACH',
    'STANDING_VERTICAL_LEAP', 'MAX_VERTICAL_LEAP',
    'LANE_AGILITY_TIME', 'THREE_QUARTER_SPRINT',
    'BODY_FAT_PCT', 'HAND_LENGTH', 'HAND_WIDTH', 'BENCH_PRESS'
]

nulos_combine = combine[cols_fisicas].isnull().sum()
pct_nulos = (nulos_combine / len(combine) * 100).round(1)
resumen = pd.DataFrame({'nulos': nulos_combine, '% nulos': pct_nulos})
print(resumen.to_string())

# Columnas de tiro — inutilizables
cols_tiro = [c for c in combine.columns if any(x in c for x in ['SPOT', 'OFF_DRIB', 'ON_MOVE'])]
print(f"\nColumnas de tiro: {len(cols_tiro)} columnas → descartadas (>90% nulos, formato string '4-5')")

### Cruce de datasets — resultado del merge

El merge se realiza por **nombre del jugador + año de draft**, normalizando a minúsculas para evitar problemas de formato.

In [ ]:
# Merge por nombre normalizado + año
combine['player_merge'] = (combine['FIRST_NAME'] + ' ' + combine['LAST_NAME']).str.lower().str.strip()
draft['player_merge'] = draft['player'].str.lower().str.strip()

merged = draft.merge(
    combine,
    left_on=['year', 'player_merge'],
    right_on=['SEASON', 'player_merge'],
    how='inner'
)

draft_2000plus = draft[draft['year'] >= 2000]
cobertura = len(merged) / len(draft_2000plus) * 100

print(f"Jugadores en draft 2000–2021:     {len(draft_2000plus)}")
print(f"Jugadores con datos de Combine:   {len(merged)}")
print(f"Cobertura del merge:              {cobertura:.1f}%")
print(f"\nDistribución por ronda:")
merged['ronda'] = merged['overall_pick'].apply(lambda x: '1ª ronda' if x <= 30 else '2ª ronda')
print(merged['ronda'].value_counts().to_string())

### Conclusiones del escaneo

- **Dataset de trabajo:** 797 jugadores con datos completos de Combine y pick real de draft.
- **Features de entrada:** 8 variables físicas y de atletismo del Combine, todas PRE-draft.
- **Target:** `overall_pick` → transformado en ronda (clasificación) y rango de picks (clasificación multiclase).
- **Sergio De Larrea:** no fue al Combine 2026. Se buscarán sus medidas físicas en fuentes externas (Basketball Reference, ACB) para incluirlo en la predicción.

---

---
## Los tres españoles en el Combine 2026

| Jugador | Posición | Altura | Peso | Envergadura |
|---------|----------|--------|------|-------------|
| Aday Mara | C | 221 cm | 117.8 kg | 229 cm |
| Baba Miller | PF | 209 cm | 94.4 kg | 218 cm |
| Sergio De Larrea | PG | 198 cm | 79 kg | 202 cm |

---

## Modelos planificados

El proyecto contempla tres enfoques complementarios:

### 1. Clasificación — ¿En qué ronda será drafteado?
**Target: saber si entrarán en  1ª ronda, en 2ª ronda o si puede que no sean drafteados.

### 2. Regresión — ¿Qué pick exacto?
**Target:** para aquellos jugadores cuya predicción sea ser drafteado, ¿qué número de elección tendrán?

### 3. No supervisado — ¿A qué perfil pertenece? (idea a desarrollar)

Se agrupa automáticamente a todos los jugadores históricos en clusters según sus medidas físicas y rendimiento. Es decir, además de predecir ronda y nº de elección, también podríamos agruparlos en categorías. Por ejemplo, Aday Mara pertenece al cluster de pívots europeos tipo Wembanyama o Jokic. Puede haber categorías como bases rápidos y ligeros, aleros largos y atléticos, pívots dominantes, etc.

---

## ¿Cuál es el objetivo final?

- Crear una **app interactiva** donde introducir las medidas de cualquier jugador y obtener una predicción de su pick en el Draft.
- Un **análisis completo** sobre las posibilidades reales de Aday Mara, Baba Miller y Sergio De Larrea en el Draft 2026, respaldado por Machine Learning.
- Acercar el mundo del periodismo a herramientas tecnológicas poco utilizadas por los profesionales del sector.